Imports

In [1]:
%load_ext autoreload
%autoreload 2

Data

In [2]:
import pandas as pd
import numpy as np

# Buat comparison
from sklearn.naive_bayes import GaussianNB   
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression as LG
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

X_train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

X_train

,id,num_adults,num_children,weekend_nights,weekday_nights,meal_package,needs_parking,room_category,days_before_arrival,arrival_year,arrival_month,arrival_day,booking_channel,is_returning_guest,prior_cancellations,prior_completed_bookings,nightly_rate_idr,special_requests,is_canceled
0,57152f82,2.0,NaN,1,2,Breakfast,0.0,Type B,88.0,2024,4,3,Offline Agent,0,0,0,1330000.0,0.0,1
1,af2ec71b,2.0,0.0,2,1,Half Board,0.0,Type D,120.0,2023,8,1,Offline Agent,0,0,0,NaN,0.0,0
2,5d5469ab,3.0,0.0,1,2,Breakfast,0.0,Type G,209.0,2024,8,28,Online Travel Agent,0,0,0,2099000.0,0.0,1
3,d2df0369,2.0,1.0,2,1,Breakfast,0.0,Type G,93.0,2024,8,13,Online Travel Agent,0,0,0,1693000.0,2.0,0
4,91d253db,2.0,0.0,0,1,Room Only,0.0,Type B,14.0,2024,9,1,Online Travel Agent,0,0,0,2010000.0,2.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27368,60213561,2.0,0.0,2,0,Room Only,0.0,Type B,2.0,2024,8,27,Online Travel Agent,0,0,0,1429000.0,0.0,0
27369,15354b3d,2.0,0.0,2,2,Room Only,0.0,Type B,NaN,2024,12,22,Online Travel Agent,0,0,0,1205000.0,1.0,1
27370,530d530c,2.0,0.0,0,2,Breakfast,0.0,Type C,115.0,2024,4,28,Online Travel Agent,0,0,0,1795000.0,0.0,1
27371,1d5d9c6a,2.0,NaN,0,1,Breakfast,0.0,Type B,58.0,2024,4,5,Offline Agent,0,0,0,1305000.0,0.0,0


Library

In [3]:
from kagglemethis.DTL import DTL
from kagglemethis.LogisticRegression import LogisticRegression
from kagglemethis.NaiveBayes import NaiveBayes
from kagglemethis.KNN import KNN

Pre Processing - lowkey buns masihan

In [4]:
X = X_train.drop(columns=["id", "is_canceled"])
y = X_train["is_canceled"]

cat_cols = X.select_dtypes(exclude=np.number).columns
num_cols = X.select_dtypes(include=np.number).columns

medians = X[num_cols].median()

# train pre process
X[cat_cols] = X[cat_cols].fillna("Missing")
X[num_cols] = X[num_cols].fillna(medians)
X = pd.get_dummies(X, columns=cat_cols, dtype=int)

# test pre process
X_test = test.drop(columns=["id"])
X_test[cat_cols] = X_test[cat_cols].fillna("Missing")
X_test[num_cols] = X_test[num_cols].fillna(medians)
X_test = pd.get_dummies(X_test, columns=cat_cols, dtype=int)
X_test = X_test.reindex(columns=X.columns, fill_value=0)


# scaling
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print(X.shape)
print(y.shape)
print(X_test.shape)

(27373, 32)
(27373,)
(6844, 32)


Model Declaration & Initial Accuracy 

In [ ]:
dtl = DTL(max_depth=5)
dtl.fit(X, y)
dtl_scikit = DecisionTreeClassifier()
dtl_scikit.fit(X,y)



logistic = LogisticRegression()
logistic.fit(X, y)
logistic_scikit = LG()
logistic_scikit.fit(X,y)

naive_bayes = NaiveBayes()
naive_bayes.fit(X,y,categorical_features=[4,5,6,11,12])
naive_scikit = GaussianNB()
naive_scikit.fit(X, y)

knn = KNN(k=5, metric="euclidean")
knn.fit(X, y)
knn_scikit = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
knn_scikit.fit(X, y)


print("Training accuracy DTL:", dtl.score(X, y))
print("Training accuracy DTL-scikit", dtl_scikit.score(X,y))
print("Training accuracy LR:", logistic.score(X, y))
print("Training accuracy LR-scikit:", logistic_scikit.score(X, y))
print("Training accuracy NB:", naive_bayes.score(X, y))
print("Training accuracy NB-scikit:", naive_scikit.score(X, y))
print("Training accuracy NB:", knn.score(X, y))
print("Training accuracy KNN-scikit:", knn_scikit.score(X, y))

Training accuracy DTL: 0.8232930259745004
Training accuracy DTL-scikit 0.9998538706024185
Training accuracy LR: 0.7816096153143609
Training accuracy LR-scikit: 0.7990355459759617
Training accuracy NB: 0.39235743250648447
Training accuracy NB-scikit: 0.4080298103971066


Submit

In [ ]:
model = dtl 

def submit(model, X_test, test):
    predictions = model.predict(X_test)

    result = pd.DataFrame({
        "id": test["id"],
        "is_canceled": predictions
    })

    result.to_csv("data/submission.csv", index=False)
    return result.head()

submit(model, X_test, test)